# CSBS-BS Blind Scoring Clinician Assignment Algorithm

**Goal.** For each completed CSBS-BS visit, assign one scoring clinician from $C = \{\text{Emma}, \text{Tessa}, \text{Axie}\}$ such that:

1. the scoring clinician is never the same person as the visit examiner;
2. the scoring clinician is as blind as possible to that child; and
3. workload stays reasonably balanced whenever blindness alone does not break the tie.

**Important interpretation used in this notebook.**

- A clinician's "contact" with a child means **any** CSBS examiner visit for that child anywhere in the dataset, not only earlier visits.
- A row with `Examiner = NA` gets `Assigned Scoring Clinician = NA`.
- Workload balance is a **tie-breaker**, not the main objective.


1. Prefer a clinician who has never examined that child.
2. If nobody is fully blind, prefer the clinician who has examined that child the fewest times.
3. If there is still a tie, prefer the clinician whose own visit with that child is furthest away in month/age from the visit being scored.
4. If there is still a tie, choose the clinician with the lower current scoring workload.
5. If everything is still tied, break the tie at random.

## 1. Mathematical Setup

Let the full dataset be a collection of visit rows

$$
\mathcal{D} = \{r_t\}_{t=1}^{N},
\qquad
r_t = (i_t, m_t, e_t),
$$

where:

- $i_t$ = child ID,
- $m_t$ = visit month (for example 9, 12, 15, 18, 24),
- $e_t \in C \cup \{\mathrm{NA}\}$ = examiner recorded for that visit.

### Core Notation

For each child $i$ and clinician $c \in C$, define

$$
V(c, i) = \{m : (i, m, c) \in \mathcal{D}\},
$$

which is the set of visit months where clinician $c$ served as examiner for child $i$.

The total examiner-contact count is

$$
n(c, i) = \lvert V(c, i) \rvert.
$$

For a completed visit row $r_t = (i_t, m_t, e_t)$, the eligible scoring pool is

$$
\mathrm{Cand}(r_t) = C \setminus \{e_t\}.
$$

Because there are only three clinicians, every completed visit has at most two eligible scoring candidates:

$$
\lvert \mathrm{Cand}(r_t) \rvert \leq 2.
$$

Rows are processed in deterministic order, by ascending `ID` and ascending `Visit Month`.

The running workload for clinician $c$ before assigning row $t$ is

$$
W_t(c) = \text{number of scoring assignments already given to clinician } c \text{ before row } t.
$$

### One-Line Formula Sheet

$$
\begin{aligned}
\mathcal{D} &= \{r_t\}_{t=1}^{N}, & r_t &= (i_t, m_t, e_t), \\
V(c,i) &= \{m : (i,m,c) \in \mathcal{D}\}, & n(c,i) &= \lvert V(c,i) \rvert, \\
\mathrm{Cand}(r_t) &= C \setminus \{e_t\}, & W_t(c) &= \text{running workload before row } t.
\end{aligned}
$$

## 2. Decision Rules

### Step 1. Never Seen (NS)

A clinician who has never examined this child is the strongest candidate:

$$
\mathrm{NS}(r_t) = \{c \in \mathrm{Cand}(r_t) : n(c, i_t) = 0\}.
$$

Decision rule:

- If $\lvert \mathrm{NS}(r_t) \rvert = 1$, assign that clinician.
- If $\lvert \mathrm{NS}(r_t) \rvert = 2$, assign

$$
c^*(r_t) = \operatorname*{argmin}_{c \in \mathrm{NS}(r_t)} W_t(c),
$$

and if workloads are still tied, break the tie at random.

### Step 2. Least Visits (LV)

If nobody is fully blind, prefer the clinician who has examined that child the fewest times:

$$
n^*(r_t) = \min_{c \in \mathrm{Cand}(r_t)} n(c, i_t),
$$

$$
\mathrm{LV}(r_t) = \{c \in \mathrm{Cand}(r_t) : n(c, i_t) = n^*(r_t)\}.
$$

Decision rule:

- If $\lvert \mathrm{LV}(r_t) \rvert = 1$, assign that clinician.
- If $\lvert \mathrm{LV}(r_t) \rvert = 2$, continue to Step 3.

### Step 3. Furthest in Time (FT)

When two clinicians are tied on contact count, compare how close their nearest own visit is to the visit being scored. For a tied clinician $c \in \mathrm{LV}(r_t)$, define

$$
d(c \mid r_t) = \min_{v \in V(c, i_t)} \lvert m_t - v \rvert.
$$

A larger distance is better, because it means the closest age/month at which that clinician saw the child is further away from the visit being scored.

Decision rule:

$$
c^*(r_t) = \operatorname*{argmax}_{c \in \mathrm{LV}(r_t)} d(c \mid r_t).
$$

If there is still a tie in $d(c \mid r_t)$, break it by lower workload $W_t(c)$, and randomize only if workload is also tied.

### Step 4. Missing Visits

If $e_t = \mathrm{NA}$, then no CSBS visit occurred and no scoring clinician is assigned:

$$
c^*(r_t) = \mathrm{NA}.
$$

That row does not change any $n(c,i)$ values and does not increase workload.

### Compact Decision Ladder

$$
\boxed{
\text{Never Seen} \;\rightarrow\; \text{Least Visits} \;\rightarrow\; \text{Furthest in Time} \;\rightarrow\; \text{Lower Workload} \;\rightarrow\; \text{Random Tie-Break}
}
$$

## 2A. Compact Mathematical Form and Scorecard

The full rule set can be written as a lexicographic optimization problem. For each completed visit row $r_t$, define the candidate score vector

$$
S(c \mid r_t) = \Bigl(\mathbb{1}\{n(c,i_t) > 0\},\; n(c,i_t),\; -d(c \mid r_t),\; W_t(c)\Bigr),
\qquad c \in \mathrm{Cand}(r_t).
$$

Interpretation of the four score components:

- $\mathbb{1}\{n(c,i_t) > 0\} = 0$ means the clinician has never seen that child.
- Smaller $n(c,i_t)$ is better.
- Larger $d(c \mid r_t)$ is better, so the score uses $-d(c \mid r_t)$.
- Smaller workload $W_t(c)$ is better.

Then the assignment rule is

$$
c^*(r_t) = \operatorname*{lexmin}_{c \in \mathrm{Cand}(r_t)} S(c \mid r_t).
$$

Equivalently,

$$
c^*(r_t) = \operatorname*{lexmin}_{c \in \mathrm{Cand}(r_t)}
\Bigl(
\mathbb{1}\{n(c,i_t) > 0\},
\; n(c,i_t),
\; -d(c \mid r_t),
\; W_t(c)
\Bigr).
$$

This exactly matches the verbal rule order: first blindness, then fewest contacts, then furthest-in-time separation, then workload balance.

### Scorecard Used in the Step Tables

| Quantity | Formula | Meaning |
| --- | --- | --- |
| Candidate set | $\mathrm{Cand}(r_t) = C \setminus \{e_t\}$ | Everyone except the current examiner |
| Never-seen flag | $\mathbb{1}\{n(c,i_t)=0\}$ | Best possible blindness |
| Contact count | $n(c,i_t)=\lvert V(c,i_t) \rvert$ | Number of examiner contacts for that child |
| Nearest-gap distance | $d(c \mid r_t)=\min_{v \in V(c,i_t)} \lvert m_t-v \rvert$ | Distance from the scored visit to the clinician's closest own visit |
| Workload | $W_t(c)$ | Running number of scoring assignments before row $t$ |

The synthetic tables below show these quantities row by row so the winner can be checked directly.

## 3. Executable Implementation

The Python cells below implement the rule set exactly as written above.

**Inputs expected by the function**

- `ID`
- `Visit Month`
- `Examiner`

**Outputs returned by the function**

- `Assigned Scoring Clinician`
- `Assignment Rule`
- `Assigned Clinician Visit Count`
- `Assigned Clinician Nearest Gap`
- `Workload Before Assignment`

The implementation is deterministic except for a true final tie, in which case a random seed is used so the result is reproducible.

In [19]:
import random
from collections import Counter

import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)

CLINICIANS = ("Emma", "Tessa", "Axie")

In [20]:
def _canonicalize_examiner(value, clinicians=CLINICIANS):
    if pd.isna(value):
        return "NA"

    text = str(value).strip()
    if not text or text.casefold() == "na":
        return "NA"

    allowed = {clinician.casefold(): clinician for clinician in clinicians}
    if text.casefold() not in allowed:
        raise ValueError(f"Unexpected examiner value: {value!r}")
    return allowed[text.casefold()]


def _nearest_gap(month, months):
    if not months:
        return pd.NA
    return min(abs(month - visited_month) for visited_month in months)


def _choose_lowest_workload(candidates, workload, rng):
    min_workload = min(workload[clinician] for clinician in candidates)
    finalists = sorted(
        clinician for clinician in candidates if workload[clinician] == min_workload
    )
    if len(finalists) == 1:
        return finalists[0], "lower workload"
    return rng.choice(finalists), "random after workload tie"


def assign_scoring_clinicians(
    df,
    clinicians=CLINICIANS,
    id_col="ID",
    month_col="Visit Month",
    examiner_col="Examiner",
    seed=42,
    return_trace=False,
):
    required_columns = {id_col, month_col, examiner_col}
    missing_columns = required_columns.difference(df.columns)
    if missing_columns:
        missing_text = ", ".join(sorted(missing_columns))
        raise KeyError(f"Missing required columns: {missing_text}")

    result = df.copy()
    result[examiner_col] = result[examiner_col].map(
        lambda value: _canonicalize_examiner(value, clinicians)
    )
    result["_sort_month"] = pd.to_numeric(result[month_col], errors="raise")
    result = result.sort_values([id_col, "_sort_month"], kind="stable").reset_index(drop=True)

    completed_visits = result[result[examiner_col] != "NA"]
    visit_history = {
        (child_id, clinician): sorted(group["_sort_month"].tolist())
        for (child_id, clinician), group in completed_visits.groupby([id_col, examiner_col], sort=False)
    }

    rng = random.Random(seed)
    workload = Counter({clinician: 0 for clinician in clinicians})

    assigned = []
    assignment_rules = []
    assigned_counts = []
    assigned_gaps = []
    workload_before_assignment = []
    trace_rows = []
    trace_columns = [
        id_col,
        month_col,
        examiner_col,
        "Candidate",
        "Candidate Visit Count n(c,i)",
        "Never Seen NS",
        "Least Visits LV",
        "Furthest in Time FT",
        "Nearest Gap d(c|r_t)",
        "Workload Before W_t(c)",
        "Selected",
        "Winning Rule",
    ]

    for _, row in result.iterrows():
        child_id = row[id_col]
        month = row["_sort_month"]
        examiner = row[examiner_col]

        if examiner == "NA":
            assigned.append("NA")
            assignment_rules.append("NA visit")
            assigned_counts.append(pd.NA)
            assigned_gaps.append(pd.NA)
            workload_before_assignment.append(pd.NA)
            continue

        candidates = [clinician for clinician in clinicians if clinician != examiner]
        candidate_counts = {
            clinician: len(visit_history.get((child_id, clinician), []))
            for clinician in candidates
        }
        candidate_gaps = {
            clinician: _nearest_gap(month, visit_history.get((child_id, clinician), []))
            for clinician in candidates
        }
        never_seen = [
            clinician for clinician, count in candidate_counts.items() if count == 0
        ]
        least_visits = []
        furthest_candidates = []

        if never_seen:
            if len(never_seen) == 1:
                chosen = never_seen[0]
                rule = "NS"
            else:
                chosen, tie_reason = _choose_lowest_workload(never_seen, workload, rng)
                rule = f"NS + {tie_reason}"
        else:
            min_count = min(candidate_counts.values())
            least_visits = [
                clinician
                for clinician, count in candidate_counts.items()
                if count == min_count
            ]

            if len(least_visits) == 1:
                chosen = least_visits[0]
                rule = "LV"
            else:
                max_gap = max(candidate_gaps[clinician] for clinician in least_visits)
                furthest_candidates = [
                    clinician
                    for clinician in least_visits
                    if candidate_gaps[clinician] == max_gap
                ]

                if len(furthest_candidates) == 1:
                    chosen = furthest_candidates[0]
                    rule = "FT"
                else:
                    chosen, tie_reason = _choose_lowest_workload(
                        furthest_candidates,
                        workload,
                        rng,
                    )
                    rule = f"FT + {tie_reason}"

        assigned.append(chosen)
        assignment_rules.append(rule)
        assigned_counts.append(candidate_counts[chosen])
        assigned_gaps.append(candidate_gaps[chosen])
        workload_before_assignment.append(workload[chosen])

        for clinician in candidates:
            trace_rows.append(
                {
                    id_col: child_id,
                    month_col: row[month_col],
                    examiner_col: examiner,
                    "Candidate": clinician,
                    "Candidate Visit Count n(c,i)": candidate_counts[clinician],
                    "Never Seen NS": clinician in never_seen,
                    "Least Visits LV": clinician in least_visits,
                    "Furthest in Time FT": clinician in furthest_candidates,
                    "Nearest Gap d(c|r_t)": candidate_gaps[clinician],
                    "Workload Before W_t(c)": workload[clinician],
                    "Selected": clinician == chosen,
                    "Winning Rule": rule,
                }
            )

        workload[chosen] += 1

    result["Assigned Scoring Clinician"] = assigned
    result["Assignment Rule"] = assignment_rules
    result["Assigned Clinician Visit Count"] = assigned_counts
    result["Assigned Clinician Nearest Gap"] = assigned_gaps
    result["Workload Before Assignment"] = workload_before_assignment
    result = result.drop(columns=["_sort_month"])

    if return_trace:
        trace_df = pd.DataFrame(trace_rows, columns=trace_columns)
        return result, trace_df

    return result


def workload_summary(assigned_df):
    completed = assigned_df[assigned_df["Assigned Scoring Clinician"] != "NA"]
    if completed.empty:
        return pd.Series(dtype="int64")
    return (
        completed["Assigned Scoring Clinician"]
        .value_counts()
        .rename_axis("Clinician")
        .to_frame("Assignments")
    )

In [21]:
example_df = pd.DataFrame(
    [
        {"ID": 1108, "Visit Month": 9, "Examiner": "Tessa"},
        {"ID": 1108, "Visit Month": 12, "Examiner": "Tessa"},
        {"ID": 1108, "Visit Month": 15, "Examiner": "Emma"},
        {"ID": 1108, "Visit Month": 18, "Examiner": "Tessa"},
        {"ID": 1108, "Visit Month": 24, "Examiner": "Emma"},
        {"ID": 1120, "Visit Month": 9, "Examiner": "Axie"},
        {"ID": 1120, "Visit Month": 12, "Examiner": "Tessa"},
        {"ID": 1120, "Visit Month": 15, "Examiner": "NA"},
        {"ID": 1120, "Visit Month": 18, "Examiner": "Tessa"},
        {"ID": 1120, "Visit Month": 24, "Examiner": "Emma"},
    ]
)

assigned_examples = assign_scoring_clinicians(example_df, seed=42)
assigned_examples

,ID,Visit Month,Examiner,Assigned Scoring Clinician,Assignment Rule,Assigned Clinician Visit Count,Assigned Clinician Nearest Gap,Workload Before Assignment
0,1108,9,Tessa,Axie,NS,0,<NA>,0
1,1108,12,Tessa,Axie,NS,0,<NA>,1
2,1108,15,Emma,Axie,NS,0,<NA>,2
3,1108,18,Tessa,Axie,NS,0,<NA>,3
4,1108,24,Emma,Axie,NS,0,<NA>,4
5,1120,9,Axie,Emma,LV,1,15,0
6,1120,12,Tessa,Emma,FT,1,12,1
7,1120,15,NA,NA,NA visit,<NA>,<NA>,<NA>
8,1120,18,Tessa,Axie,FT,1,9,5
9,1120,24,Emma,Axie,LV,1,15,6


## 4. Synthetic Showcase Tables

The next cells use a synthetic dataset designed to force each main branch of the algorithm:

| Child ID | Intended scenario | What should happen |
| --- | --- | --- |
| 2101 | Never Seen (NS) | Axie has zero examiner-contact for the child, so Axie should be assigned throughout |
| 2102 | Least Visits (LV) and Furthest in Time (FT) | All clinicians have seen the child at least once, so counts and month gaps determine the scorer |
| 2103 | NS tie broken by workload | Two clinicians are equally blind, so the lower current workload should win |

In the candidate tables:

- `Candidate Visit Count n(c,i)` corresponds to $n(c,i)$,
- `Nearest Gap d(c|r_t)` corresponds to $d(c \mid r_t)$,
- `Workload Before W_t(c)` corresponds to $W_t(c)$.

A row is selected when it minimizes the lexicographic score

$$
\Bigl(\mathbb{1}\{n(c,i_t) > 0\},\; n(c,i_t),\; -d(c \mid r_t),\; W_t(c)\Bigr).
$$

In [22]:
synthetic_showcase_df = pd.DataFrame(
    [
        {"ID": 2101, "Visit Month": 9, "Examiner": "Tessa"},
        {"ID": 2101, "Visit Month": 12, "Examiner": "Tessa"},
        {"ID": 2101, "Visit Month": 15, "Examiner": "Emma"},
        {"ID": 2101, "Visit Month": 18, "Examiner": "Tessa"},
        {"ID": 2101, "Visit Month": 24, "Examiner": "Emma"},
        {"ID": 2102, "Visit Month": 9, "Examiner": "Axie"},
        {"ID": 2102, "Visit Month": 12, "Examiner": "Tessa"},
        {"ID": 2102, "Visit Month": 15, "Examiner": "NA"},
        {"ID": 2102, "Visit Month": 18, "Examiner": "Tessa"},
        {"ID": 2102, "Visit Month": 24, "Examiner": "Emma"},
        {"ID": 2103, "Visit Month": 9, "Examiner": "Emma"},
    ]
)

synthetic_assigned, synthetic_trace = assign_scoring_clinicians(
    synthetic_showcase_df,
    seed=42,
    return_trace=True,
    )

synthetic_assigned[
    [
        "ID",
        "Visit Month",
        "Examiner",
        "Assigned Scoring Clinician",
        "Assignment Rule",
    ]
]

,ID,Visit Month,Examiner,Assigned Scoring Clinician,Assignment Rule
0,2101,9,Tessa,Axie,NS
1,2101,12,Tessa,Axie,NS
2,2101,15,Emma,Axie,NS
3,2101,18,Tessa,Axie,NS
4,2101,24,Emma,Axie,NS
5,2102,9,Axie,Emma,LV
6,2102,12,Tessa,Emma,FT
7,2102,15,NA,NA,NA visit
8,2102,18,Tessa,Axie,FT
9,2102,24,Emma,Axie,LV


In [23]:
for child_id, scenario in [
    (2101, "NS showcase"),
    (2102, "LV and FT showcase"),
    (2103, "NS + workload showcase"),
]:
    print(f"Child {child_id} — {scenario}")
    display(
        synthetic_trace.loc[synthetic_trace["ID"] == child_id, [
            "Visit Month",
            "Examiner",
            "Candidate",
            "Candidate Visit Count n(c,i)",
            "Never Seen NS",
            "Least Visits LV",
            "Furthest in Time FT",
            "Nearest Gap d(c|r_t)",
            "Workload Before W_t(c)",
            "Selected",
            "Winning Rule",
        ]].reset_index(drop=True)
    )

workload_summary(synthetic_assigned)

Child 2101 — NS showcase


,Visit Month,Examiner,Candidate,"Candidate Visit Count n(c,i)",Never Seen NS,Least Visits LV,Furthest in Time FT,Nearest Gap d(c|r_t),Workload Before W_t(c),Selected,Winning Rule
0,9,Tessa,Emma,2,False,False,False,6,0,False,NS
1,9,Tessa,Axie,0,True,False,False,<NA>,0,True,NS
2,12,Tessa,Emma,2,False,False,False,3,0,False,NS
3,12,Tessa,Axie,0,True,False,False,<NA>,1,True,NS
4,15,Emma,Tessa,3,False,False,False,3,0,False,NS
5,15,Emma,Axie,0,True,False,False,<NA>,2,True,NS
6,18,Tessa,Emma,2,False,False,False,3,0,False,NS
7,18,Tessa,Axie,0,True,False,False,<NA>,3,True,NS
8,24,Emma,Tessa,3,False,False,False,6,0,False,NS
9,24,Emma,Axie,0,True,False,False,<NA>,4,True,NS


Child 2102 — LV and FT showcase


,Visit Month,Examiner,Candidate,"Candidate Visit Count n(c,i)",Never Seen NS,Least Visits LV,Furthest in Time FT,Nearest Gap d(c|r_t),Workload Before W_t(c),Selected,Winning Rule
0,9,Axie,Emma,1,False,True,False,15,0,True,LV
1,9,Axie,Tessa,2,False,False,False,3,0,False,LV
2,12,Tessa,Emma,1,False,True,True,12,1,True,FT
3,12,Tessa,Axie,1,False,True,False,3,5,False,FT
4,18,Tessa,Emma,1,False,True,False,6,2,False,FT
5,18,Tessa,Axie,1,False,True,True,9,5,True,FT
6,24,Emma,Tessa,2,False,False,False,6,0,False,LV
7,24,Emma,Axie,1,False,True,False,15,6,True,LV


Child 2103 — NS + workload showcase


,Visit Month,Examiner,Candidate,"Candidate Visit Count n(c,i)",Never Seen NS,Least Visits LV,Furthest in Time FT,Nearest Gap d(c|r_t),Workload Before W_t(c),Selected,Winning Rule
0,9,Emma,Tessa,0,True,False,False,<NA>,0,True,NS + lower workload
1,9,Emma,Axie,0,True,False,False,<NA>,7,False,NS + lower workload


,Assignments
Clinician,
Axie,7
Emma,2
Tessa,1


## 5. Validation Against the Original Worked Examples

The next cell reruns the two original worked examples and checks that the assignments still match the expected clinician choices exactly.

In [24]:
expected_assignments = {
    (1108, 9): "Axie",
    (1108, 12): "Axie",
    (1108, 15): "Axie",
    (1108, 18): "Axie",
    (1108, 24): "Axie",
    (1120, 9): "Emma",
    (1120, 12): "Emma",
    (1120, 15): "NA",
    (1120, 18): "Axie",
    (1120, 24): "Axie",
}

validation_df = assigned_examples[
    ["ID", "Visit Month", "Examiner", "Assigned Scoring Clinician", "Assignment Rule"]
].copy()
validation_df["Expected"] = validation_df.apply(
    lambda row: expected_assignments[(row["ID"], row["Visit Month"])],
    axis=1,
)
validation_df["Matches Expected"] = (
    validation_df["Assigned Scoring Clinician"] == validation_df["Expected"]
)

completed_visits = assigned_examples[assigned_examples["Examiner"] != "NA"]
na_visits = assigned_examples[assigned_examples["Examiner"] == "NA"]
valid_assignments = set(CLINICIANS) | {"NA"}

assert set(assigned_examples["Assigned Scoring Clinician"]).issubset(valid_assignments)
assert (completed_visits["Assigned Scoring Clinician"] != completed_visits["Examiner"]).all()
assert na_visits["Assigned Scoring Clinician"].eq("NA").all()
assert validation_df["Matches Expected"].all()

display(validation_df)
print("All validation checks passed.")
workload_summary(assigned_examples)

,ID,Visit Month,Examiner,Assigned Scoring Clinician,Assignment Rule,Expected,Matches Expected
0,1108,9,Tessa,Axie,NS,Axie,True
1,1108,12,Tessa,Axie,NS,Axie,True
2,1108,15,Emma,Axie,NS,Axie,True
3,1108,18,Tessa,Axie,NS,Axie,True
4,1108,24,Emma,Axie,NS,Axie,True
5,1120,9,Axie,Emma,LV,Emma,True
6,1120,12,Tessa,Emma,FT,Emma,True
7,1120,15,NA,NA,NA visit,NA,True
8,1120,18,Tessa,Axie,FT,Axie,True
9,1120,24,Emma,Axie,LV,Axie,True


All validation checks passed.


,Assignments
Clinician,
Axie,7
Emma,2


## 6. Worked Example Calculations and Usage Notes

### Child 1108

Across the full dataset for child 1108,

$$
\bigl(n(\text{Emma}, 1108),\; n(\text{Tessa}, 1108),\; n(\text{Axie}, 1108)\bigr) = (2, 3, 0).
$$

So for every completed visit, Axie is the unique clinician with zero examiner-contact for that child. Therefore

$$
\mathrm{NS}(r_t) = \{\text{Axie}\}.
$$

Hence Axie is assigned for every completed 1108 row.

### Child 1120

Across the full dataset for child 1120,

$$
\bigl(n(\text{Emma}, 1120),\; n(\text{Tessa}, 1120),\; n(\text{Axie}, 1120)\bigr) = (1, 2, 1).
$$

That leads to the following row-level logic:

- **9m, examiner = Axie:** candidates are Emma and Tessa. Since $n(\text{Emma},1120)=1 < 2=n(\text{Tessa},1120)$, the LV rule assigns Emma.
- **12m, examiner = Tessa:** candidates are Emma and Axie, both with one prior examiner-contact. Compare nearest month gaps:

$$
d(\text{Emma} \mid 12) = \lvert 12 - 24 \rvert = 12,
\qquad
d(\text{Axie} \mid 12) = \lvert 12 - 9 \rvert = 3.
$$

Emma is further away in time, so the FT rule assigns Emma.

- **15m, examiner = NA:** no assignment is made.
- **18m, examiner = Tessa:** Emma and Axie are tied on contact count, so compare

$$
d(\text{Emma} \mid 18) = \lvert 18 - 24 \rvert = 6,
\qquad
d(\text{Axie} \mid 18) = \lvert 18 - 9 \rvert = 9.
$$

Axie is further away in time, so the FT rule assigns Axie.

- **24m, examiner = Emma:** candidates are Axie and Tessa. Since $n(\text{Axie},1120)=1 < 2=n(\text{Tessa},1120)$, the LV rule assigns Axie.

### Validation and Reuse

The validation cells in this notebook check four things:

1. the two worked examples produce the expected assignments;
2. no assigned scoring clinician is ever the same as the examiner;
3. rows with `Examiner = NA` remain unassigned; and
4. workload totals can be reviewed afterward to see whether balance looks reasonable.

For real data, the intended call is simply:

```python
assigned_df = assign_scoring_clinicians(your_df)
```

where `your_df` contains the columns `ID`, `Visit Month`, and `Examiner`.

## 7. Plain-English Summary from the Synthetic Showcase

The final cell below reads the synthetic assignment outputs and converts them into a short narrative summary automatically.

In [25]:
from IPython.display import Markdown, display

def _rule_count(frame, prefix):
    return int(frame["Assignment Rule"].fillna("").str.startswith(prefix).sum())

completed_synthetic = synthetic_assigned[synthetic_assigned["Examiner"] != "NA"].copy()
total_rows = len(synthetic_assigned)
completed_rows = len(completed_synthetic)
na_rows = int((synthetic_assigned["Examiner"] == "NA").sum())
ns_rows = _rule_count(completed_synthetic, "NS")
lv_rows = _rule_count(completed_synthetic, "LV")
ft_rows = _rule_count(completed_synthetic, "FT")

child_2101_assignee = completed_synthetic.loc[
    completed_synthetic["ID"] == 2101, "Assigned Scoring Clinician"
].unique().tolist()
child_2102_rules = completed_synthetic.loc[
    completed_synthetic["ID"] == 2102, "Assignment Rule"
].tolist()
child_2103_choice = completed_synthetic.loc[
    completed_synthetic["ID"] == 2103, "Assigned Scoring Clinician"
].iloc[0]

synthetic_workload = workload_summary(synthetic_assigned).reset_index()
workload_lines = [
    f"- {row['Clinician']}: {int(row['Assignments'])} assignment(s)"
    for _, row in synthetic_workload.iterrows()
 ]

summary_markdown = "\n".join([
    "## Synthetic Showcase Findings",
    "",
    f"The synthetic dataset contains **{total_rows}** rows, of which **{completed_rows}** are completed visits and **{na_rows}** is an `NA` visit.",
    "",
    "### What the synthetic examples demonstrate",
    "",
    f"- **Never Seen (NS)** was used on **{ns_rows}** completed rows.",
    f"- **Least Visits (LV)** was used on **{lv_rows}** completed rows.",
    f"- **Furthest in Time (FT)** was used on **{ft_rows}** completed rows.",
    f"- For child **2101**, the scorer was always **{', '.join(child_2101_assignee)}**, which shows the pure NS case.",
    f"- For child **2102**, the winning rules across completed visits were **{', '.join(child_2102_rules)}**, which shows how LV and FT work together.",
    f"- For child **2103**, the chosen scorer was **{child_2103_choice}**, showing that an NS tie can be broken by current workload.",
    "",
    "### Workload pattern in the synthetic showcase",
    "",
    *workload_lines,
    "",
    "### Bottom line",
    "",
    "The synthetic examples confirm that the algorithm first protects blindness, then uses contact counts, then time separation, and only then uses workload to break remaining ties.",
])

display(Markdown(summary_markdown))

## Synthetic Showcase Findings

The synthetic dataset contains **11** rows, of which **10** are completed visits and **1** is an `NA` visit.

### What the synthetic examples demonstrate

- **Never Seen (NS)** was used on **6** completed rows.
- **Least Visits (LV)** was used on **2** completed rows.
- **Furthest in Time (FT)** was used on **2** completed rows.
- For child **2101**, the scorer was always **Axie**, which shows the pure NS case.
- For child **2102**, the winning rules across completed visits were **LV, FT, FT, LV**, which shows how LV and FT work together.
- For child **2103**, the chosen scorer was **Tessa**, showing that an NS tie can be broken by current workload.

### Workload pattern in the synthetic showcase

- Axie: 7 assignment(s)
- Emma: 2 assignment(s)
- Tessa: 1 assignment(s)

### Bottom line

The synthetic examples confirm that the algorithm first protects blindness, then uses contact counts, then time separation, and only then uses workload to break remaining ties.

## 9. Case-by-Case Worked Examples for Every Decision Path

This section gives one worked example for each distinct outcome path in the algorithm.

The cases covered are:

1. `Examiner = NA` so no scorer is assigned.
2. A **unique Never Seen** clinician exists.
3. Two **Never Seen** clinicians tie, so workload breaks the tie.
4. Two **Never Seen** clinicians tie and workload is also tied, so the final choice is random.
5. No Never Seen candidate exists, but one clinician has **fewer visits** than the other.
6. Visit counts tie, so **Furthest in Time** breaks the tie.
7. Visit counts tie and time distance also ties, so workload breaks the tie.
8. Visit counts tie, time distance ties, and workload ties, so the final choice is random.

All examples below use the same notation:

$$
\mathrm{Cand}(r_t), \qquad \mathrm{NS}(r_t), \qquad \mathrm{LV}(r_t), \qquad d(c \mid r_t), \qquad c^*(r_t).
$$

In [26]:
# This cell gives one worked example for every possible decision path in the algorithm.
# Each scenario is explained with the same formulas used in the notebook:
# Cand(r_t), NS(r_t), LV(r_t), d(c | r_t), W_t(c), and c*(r_t).

from IPython.display import Markdown, display
import random

def _latex_text(name):
    return rf"\text{{{name}}}"

def _latex_set(names):
    names = list(names)
    if not names:
        return r"\varnothing"
    return r"\{" + ", ".join(_latex_text(name) for name in names) + r"\}"

def _latex_visit_set(months):
    if not months:
        return r"\varnothing"
    return r"\{" + ", ".join(str(month) for month in months) + r"\}"

def _gap_value(month, months):
    if not months:
        return None
    return min(abs(month - seen_month) for seen_month in months)

def _gap_formula(candidate, month, months):
    candidate_text = _latex_text(candidate)
    if not months:
        return rf"d({candidate_text} \mid r_t)\text{{ is undefined because }}V({candidate_text}, i_t)=\varnothing"
    terms = [rf"\lvert {month}-{seen_month} \rvert" for seen_month in months]
    gap = _gap_value(month, months)
    if len(terms) == 1:
        return rf"d({candidate_text} \mid r_t) = {terms[0]} = {gap}"
    return rf"d({candidate_text} \mid r_t) = \min({', '.join(terms)}) = {gap}"

def _evaluate_case(case):
    rng = random.Random(case.get("seed", 42))

    if case["examiner"] == "NA":
        return {
            "chosen": "NA",
            "rule": "NA visit",
            "never_seen": [],
            "least_visits": [],
            "furthest_candidates": [],
            "counts": {},
            "gaps": {},
            "visit_sets": {},
        }

    candidates = case["candidates"]
    visit_sets = {candidate: sorted(case["visit_sets"].get(candidate, [])) for candidate in candidates}
    counts = {candidate: len(visit_sets[candidate]) for candidate in candidates}
    gaps = {candidate: _gap_value(case["month"], visit_sets[candidate]) for candidate in candidates}
    workloads = case["workloads"]

    never_seen = [candidate for candidate in candidates if counts[candidate] == 0]
    least_visits = []
    furthest_candidates = []

    if never_seen:
        if len(never_seen) == 1:
            chosen = never_seen[0]
            rule = "NS"
        else:
            min_workload = min(workloads[candidate] for candidate in never_seen)
            finalists = [candidate for candidate in never_seen if workloads[candidate] == min_workload]
            if len(finalists) == 1:
                chosen = finalists[0]
                rule = "NS + lower workload"
            else:
                chosen = rng.choice(sorted(finalists))
                rule = "NS + random tie-break"
    else:
        min_count = min(counts[candidate] for candidate in candidates)
        least_visits = [candidate for candidate in candidates if counts[candidate] == min_count]
        if len(least_visits) == 1:
            chosen = least_visits[0]
            rule = "LV"
        else:
            max_gap = max(gaps[candidate] for candidate in least_visits)
            furthest_candidates = [candidate for candidate in least_visits if gaps[candidate] == max_gap]
            if len(furthest_candidates) == 1:
                chosen = furthest_candidates[0]
                rule = "FT"
            else:
                min_workload = min(workloads[candidate] for candidate in furthest_candidates)
                finalists = [candidate for candidate in furthest_candidates if workloads[candidate] == min_workload]
                if len(finalists) == 1:
                    chosen = finalists[0]
                    rule = "FT + lower workload"
                else:
                    chosen = rng.choice(sorted(finalists))
                    rule = "FT + random tie-break"

    return {
        "chosen": chosen,
        "rule": rule,
        "never_seen": never_seen,
        "least_visits": least_visits,
        "furthest_candidates": furthest_candidates,
        "counts": counts,
        "gaps": gaps,
        "visit_sets": visit_sets,
    }

casebook = [
    {
        "title": "Case 1. Missing Visit (NA)",
        "condition": "The visit did not occur, so no scoring decision should be made.",
        "examiner": "NA",
        "month": 15,
        "candidates": [],
        "visit_sets": {},
        "workloads": {},
    },
    {
        "title": "Case 2. Unique Never Seen (NS)",
        "condition": "Exactly one eligible clinician has never examined the child.",
        "examiner": "Tessa",
        "month": 18,
        "candidates": ["Emma", "Axie"],
        "visit_sets": {"Emma": [9, 24], "Axie": []},
        "workloads": {"Emma": 5, "Axie": 1},
    },
    {
        "title": "Case 3. Never Seen Tie Broken by Workload",
        "condition": "Both eligible clinicians have never seen the child, but one currently has lower workload.",
        "examiner": "Emma",
        "month": 18,
        "candidates": ["Tessa", "Axie"],
        "visit_sets": {"Tessa": [], "Axie": []},
        "workloads": {"Tessa": 1, "Axie": 4},
    },
    {
        "title": "Case 4. Never Seen Tie Broken Randomly",
        "condition": "Both eligible clinicians have never seen the child and their workloads are also tied.",
        "examiner": "Emma",
        "month": 18,
        "candidates": ["Tessa", "Axie"],
        "visit_sets": {"Tessa": [], "Axie": []},
        "workloads": {"Tessa": 2, "Axie": 2},
        "seed": 42,
    },
    {
        "title": "Case 5. Least Visits (LV) Unique Winner",
        "condition": "No eligible clinician is fully blind, but one has fewer prior examiner-visits than the other.",
        "examiner": "Axie",
        "month": 15,
        "candidates": ["Emma", "Tessa"],
        "visit_sets": {"Emma": [24], "Tessa": [9, 12, 18]},
        "workloads": {"Emma": 4, "Tessa": 2},
    },
    {
        "title": "Case 6. Furthest in Time (FT) Unique Winner",
        "condition": "Visit counts tie, so the candidate whose nearest own visit is further away in time should win.",
        "examiner": "Tessa",
        "month": 12,
        "candidates": ["Emma", "Axie"],
        "visit_sets": {"Emma": [24], "Axie": [9]},
        "workloads": {"Emma": 4, "Axie": 4},
    },
    {
        "title": "Case 7. FT Tie Broken by Workload",
        "condition": "Visit counts tie and nearest-gap distances tie, so lower workload should break the tie.",
        "examiner": "Tessa",
        "month": 18,
        "candidates": ["Emma", "Axie"],
        "visit_sets": {"Emma": [12, 30], "Axie": [24, 36]},
        "workloads": {"Emma": 1, "Axie": 5},
    },
    {
        "title": "Case 8. FT Tie Broken Randomly",
        "condition": "Visit counts tie, nearest-gap distances tie, and workloads tie, so the final choice is random.",
        "examiner": "Tessa",
        "month": 15,
        "candidates": ["Emma", "Axie"],
        "visit_sets": {"Emma": [9, 27], "Axie": [3, 21]},
        "workloads": {"Emma": 3, "Axie": 3},
        "seed": 42,
    },
]

case_rows = []
for case in casebook:
    result = _evaluate_case(case)
    case_rows.append(
        {
            "Case": case["title"],
            "Condition": case["condition"],
            "Assigned Scoring Clinician": result["chosen"],
            "Winning Rule": result["rule"],
        }
    )

display(pd.DataFrame(case_rows))

for case in casebook:
    result = _evaluate_case(case)
    title = case["title"]
    condition = case["condition"]
    examiner = case["examiner"]
    month = case["month"]
    chosen = result["chosen"]
    rule = result["rule"]
    lines = [f"### {title}", "", f"**Condition.** {condition}", ""]

    if examiner == "NA":
        lines.extend([
            "Because the examiner value is `NA`, the visit did not occur and no scoring clinician should be assigned.",
            "",
            "$$",
            r"c^*(r_t)=\mathrm{NA}",
            "$$",
            "",
            "**Answer.** The row passes through with `Assigned Scoring Clinician = NA`.",
        ])
        display(Markdown("\n".join(lines)))
        continue

    candidates = case["candidates"]
    visit_sets = result["visit_sets"]
    counts = result["counts"]
    gaps = result["gaps"]
    workloads = case["workloads"]
    never_seen = result["never_seen"]
    least_visits = result["least_visits"]
    furthest_candidates = result["furthest_candidates"]

    lines.extend([
        f"At month **{month}**, the examiner is **{examiner}**, so the eligible candidates are **{', '.join(candidates)}**.",
        "",
        "$$",
        rf"\mathrm{{Cand}}(r_t) = C \setminus \{{e_t\}} = {_latex_set(candidates)}",
        "$$",
        "",
        "The examiner-history sets are:",
        "",
        "$$",
        r"\qquad ".join(
            rf"V({_latex_text(candidate)}, i_t) = {_latex_visit_set(visit_sets[candidate])}"
            for candidate in candidates
        ),
        "$$",
        "",
        "Therefore the contact counts are:",
        "",
        "$$",
        r"\qquad ".join(
            rf"n({_latex_text(candidate)}, i_t) = {counts[candidate]}"
            for candidate in candidates
        ),
        "$$",
        "",
        "The Never Seen set is:",
        "",
        "$$",
        rf"\mathrm{{NS}}(r_t) = {_latex_set(never_seen)}",
        "$$",
    ])

    if never_seen:
        if len(never_seen) == 1:
            lines.extend([
                "",
                rf"Since $\mathrm{{NS}}(r_t)$ contains exactly one clinician, the scorer is {_latex_text(chosen)}.",
                "",
                "$$",
                rf"c^*(r_t) = {_latex_text(chosen)}",
                "$$",
            ])
        else:
            lines.extend([
                "",
                "The Never Seen set contains both candidates, so workload is used next:",
                "",
                "$$",
                r"\qquad ".join(
                    rf"W_t({_latex_text(candidate)}) = {workloads[candidate]}"
                    for candidate in never_seen
                ),
                "$$",
            ])
            if "lower workload" in rule:
                lines.extend([
                    "",
                    rf"The smaller workload belongs to {_latex_text(chosen)}, so",
                    "",
                    "$$",
                    rf"c^*(r_t) = {_latex_text(chosen)}",
                    "$$",
                ])
            else:
                lines.extend([
                    "",
                    f"The workloads are tied, so the rule reaches a perfect tie. With seed = {case.get('seed', 42)}, the reproducible random choice is **{chosen}**.",
                    "",
                    "$$",
                    rf"c^*(r_t) = {_latex_text(chosen)}",
                    "$$",
                ])
        display(Markdown("\n".join(lines)))
        continue

    min_count = min(counts[candidate] for candidate in candidates)
    lines.extend([
        "",
        "Because no candidate is fully blind, we move to the Least Visits rule:",
        "",
        "$$",
        rf"n^*(r_t) = \min({', '.join(str(counts[candidate]) for candidate in candidates)}) = {min_count}",
        "$$",
        "",
        "$$",
        rf"\mathrm{{LV}}(r_t) = {_latex_set(least_visits)}",
        "$$",
    ])

    if len(least_visits) == 1:
        lines.extend([
            "",
            rf"Since $\mathrm{{LV}}(r_t)$ contains exactly one clinician, the scorer is {_latex_text(chosen)}.",
            "",
            "$$",
            rf"c^*(r_t) = {_latex_text(chosen)}",
            "$$",
        ])
        display(Markdown("\n".join(lines)))
        continue

    lines.extend([
        "",
        "The Least Visits set is still tied, so we compare nearest-gap distances:",
        "",
        "$$",
        _gap_formula(least_visits[0], month, visit_sets[least_visits[0]]),
        "$$",
        "",
        "$$",
        _gap_formula(least_visits[1], month, visit_sets[least_visits[1]]),
        "$$",
    ])

    max_gap = max(gaps[candidate] for candidate in least_visits)
    if len(furthest_candidates) == 1:
        lines.extend([
            "",
            "Therefore the Furthest in Time set is:",
            "",
            "$$",
            rf"\mathrm{{FT}}(r_t) = {_latex_set(furthest_candidates)}",
            "$$",
            "",
            rf"The larger gap is {max_gap}, so the scorer is {_latex_text(chosen)}.",
            "",
            "$$",
            rf"c^*(r_t) = {_latex_text(chosen)}",
            "$$",
        ])
        display(Markdown("\n".join(lines)))
        continue

    lines.extend([
        "",
        "The nearest-gap distances are tied, so workload is used next:",
        "",
        "$$",
        r"\qquad ".join(
            rf"W_t({_latex_text(candidate)}) = {workloads[candidate]}"
            for candidate in furthest_candidates
        ),
        "$$",
    ])

    if "lower workload" in rule:
        lines.extend([
            "",
            rf"The smaller workload belongs to {_latex_text(chosen)}, so",
            "",
            "$$",
            rf"c^*(r_t) = {_latex_text(chosen)}",
            "$$",
        ])
    else:
        lines.extend([
            "",
            f"The workloads are also tied, so this is a perfect final tie. With seed = {case.get('seed', 42)}, the reproducible random choice is **{chosen}**.",
            "",
            "$$",
            rf"c^*(r_t) = {_latex_text(chosen)}",
            "$$",
        ])

    display(Markdown("\n".join(lines)))

,Case,Condition,Assigned Scoring Clinician,Winning Rule
0,Case 1. Missing Visit (NA),"The visit did not occur, so no scoring decisio...",NA,NA visit
1,Case 2. Unique Never Seen (NS),Exactly one eligible clinician has never exami...,Axie,NS
2,Case 3. Never Seen Tie Broken by Workload,Both eligible clinicians have never seen the c...,Tessa,NS + lower workload
3,Case 4. Never Seen Tie Broken Randomly,Both eligible clinicians have never seen the c...,Axie,NS + random tie-break
4,Case 5. Least Visits (LV) Unique Winner,"No eligible clinician is fully blind, but one ...",Emma,LV
5,Case 6. Furthest in Time (FT) Unique Winner,"Visit counts tie, so the candidate whose neare...",Emma,FT
6,Case 7. FT Tie Broken by Workload,Visit counts tie and nearest-gap distances tie...,Emma,FT + lower workload
7,Case 8. FT Tie Broken Randomly,"Visit counts tie, nearest-gap distances tie, a...",Axie,FT + random tie-break


### Case 1. Missing Visit (NA)

**Condition.** The visit did not occur, so no scoring decision should be made.

Because the examiner value is `NA`, the visit did not occur and no scoring clinician should be assigned.

$$
c^*(r_t)=\mathrm{NA}
$$

**Answer.** The row passes through with `Assigned Scoring Clinician = NA`.

### Case 2. Unique Never Seen (NS)

**Condition.** Exactly one eligible clinician has never examined the child.

At month **18**, the examiner is **Tessa**, so the eligible candidates are **Emma, Axie**.

$$
\mathrm{Cand}(r_t) = C \setminus \{e_t\} = \{\text{Emma}, \text{Axie}\}
$$

The examiner-history sets are:

$$
V(\text{Emma}, i_t) = \{9, 24\}\qquad V(\text{Axie}, i_t) = \varnothing
$$

Therefore the contact counts are:

$$
n(\text{Emma}, i_t) = 2\qquad n(\text{Axie}, i_t) = 0
$$

The Never Seen set is:

$$
\mathrm{NS}(r_t) = \{\text{Axie}\}
$$

Since $\mathrm{NS}(r_t)$ contains exactly one clinician, the scorer is \text{Axie}.

$$
c^*(r_t) = \text{Axie}
$$

### Case 3. Never Seen Tie Broken by Workload

**Condition.** Both eligible clinicians have never seen the child, but one currently has lower workload.

At month **18**, the examiner is **Emma**, so the eligible candidates are **Tessa, Axie**.

$$
\mathrm{Cand}(r_t) = C \setminus \{e_t\} = \{\text{Tessa}, \text{Axie}\}
$$

The examiner-history sets are:

$$
V(\text{Tessa}, i_t) = \varnothing\qquad V(\text{Axie}, i_t) = \varnothing
$$

Therefore the contact counts are:

$$
n(\text{Tessa}, i_t) = 0\qquad n(\text{Axie}, i_t) = 0
$$

The Never Seen set is:

$$
\mathrm{NS}(r_t) = \{\text{Tessa}, \text{Axie}\}
$$

The Never Seen set contains both candidates, so workload is used next:

$$
W_t(\text{Tessa}) = 1\qquad W_t(\text{Axie}) = 4
$$

The smaller workload belongs to \text{Tessa}, so

$$
c^*(r_t) = \text{Tessa}
$$

### Case 4. Never Seen Tie Broken Randomly

**Condition.** Both eligible clinicians have never seen the child and their workloads are also tied.

At month **18**, the examiner is **Emma**, so the eligible candidates are **Tessa, Axie**.

$$
\mathrm{Cand}(r_t) = C \setminus \{e_t\} = \{\text{Tessa}, \text{Axie}\}
$$

The examiner-history sets are:

$$
V(\text{Tessa}, i_t) = \varnothing\qquad V(\text{Axie}, i_t) = \varnothing
$$

Therefore the contact counts are:

$$
n(\text{Tessa}, i_t) = 0\qquad n(\text{Axie}, i_t) = 0
$$

The Never Seen set is:

$$
\mathrm{NS}(r_t) = \{\text{Tessa}, \text{Axie}\}
$$

The Never Seen set contains both candidates, so workload is used next:

$$
W_t(\text{Tessa}) = 2\qquad W_t(\text{Axie}) = 2
$$

The workloads are tied, so the rule reaches a perfect tie. With seed = 42, the reproducible random choice is **Axie**.

$$
c^*(r_t) = \text{Axie}
$$

### Case 5. Least Visits (LV) Unique Winner

**Condition.** No eligible clinician is fully blind, but one has fewer prior examiner-visits than the other.

At month **15**, the examiner is **Axie**, so the eligible candidates are **Emma, Tessa**.

$$
\mathrm{Cand}(r_t) = C \setminus \{e_t\} = \{\text{Emma}, \text{Tessa}\}
$$

The examiner-history sets are:

$$
V(\text{Emma}, i_t) = \{24\}\qquad V(\text{Tessa}, i_t) = \{9, 12, 18\}
$$

Therefore the contact counts are:

$$
n(\text{Emma}, i_t) = 1\qquad n(\text{Tessa}, i_t) = 3
$$

The Never Seen set is:

$$
\mathrm{NS}(r_t) = \varnothing
$$

Because no candidate is fully blind, we move to the Least Visits rule:

$$
n^*(r_t) = \min(1, 3) = 1
$$

$$
\mathrm{LV}(r_t) = \{\text{Emma}\}
$$

Since $\mathrm{LV}(r_t)$ contains exactly one clinician, the scorer is \text{Emma}.

$$
c^*(r_t) = \text{Emma}
$$

### Case 6. Furthest in Time (FT) Unique Winner

**Condition.** Visit counts tie, so the candidate whose nearest own visit is further away in time should win.

At month **12**, the examiner is **Tessa**, so the eligible candidates are **Emma, Axie**.

$$
\mathrm{Cand}(r_t) = C \setminus \{e_t\} = \{\text{Emma}, \text{Axie}\}
$$

The examiner-history sets are:

$$
V(\text{Emma}, i_t) = \{24\}\qquad V(\text{Axie}, i_t) = \{9\}
$$

Therefore the contact counts are:

$$
n(\text{Emma}, i_t) = 1\qquad n(\text{Axie}, i_t) = 1
$$

The Never Seen set is:

$$
\mathrm{NS}(r_t) = \varnothing
$$

Because no candidate is fully blind, we move to the Least Visits rule:

$$
n^*(r_t) = \min(1, 1) = 1
$$

$$
\mathrm{LV}(r_t) = \{\text{Emma}, \text{Axie}\}
$$

The Least Visits set is still tied, so we compare nearest-gap distances:

$$
d(\text{Emma} \mid r_t) = \lvert 12-24 \rvert = 12
$$

$$
d(\text{Axie} \mid r_t) = \lvert 12-9 \rvert = 3
$$

Therefore the Furthest in Time set is:

$$
\mathrm{FT}(r_t) = \{\text{Emma}\}
$$

The larger gap is 12, so the scorer is \text{Emma}.

$$
c^*(r_t) = \text{Emma}
$$

### Case 7. FT Tie Broken by Workload

**Condition.** Visit counts tie and nearest-gap distances tie, so lower workload should break the tie.

At month **18**, the examiner is **Tessa**, so the eligible candidates are **Emma, Axie**.

$$
\mathrm{Cand}(r_t) = C \setminus \{e_t\} = \{\text{Emma}, \text{Axie}\}
$$

The examiner-history sets are:

$$
V(\text{Emma}, i_t) = \{12, 30\}\qquad V(\text{Axie}, i_t) = \{24, 36\}
$$

Therefore the contact counts are:

$$
n(\text{Emma}, i_t) = 2\qquad n(\text{Axie}, i_t) = 2
$$

The Never Seen set is:

$$
\mathrm{NS}(r_t) = \varnothing
$$

Because no candidate is fully blind, we move to the Least Visits rule:

$$
n^*(r_t) = \min(2, 2) = 2
$$

$$
\mathrm{LV}(r_t) = \{\text{Emma}, \text{Axie}\}
$$

The Least Visits set is still tied, so we compare nearest-gap distances:

$$
d(\text{Emma} \mid r_t) = \min(\lvert 18-12 \rvert, \lvert 18-30 \rvert) = 6
$$

$$
d(\text{Axie} \mid r_t) = \min(\lvert 18-24 \rvert, \lvert 18-36 \rvert) = 6
$$

The nearest-gap distances are tied, so workload is used next:

$$
W_t(\text{Emma}) = 1\qquad W_t(\text{Axie}) = 5
$$

The smaller workload belongs to \text{Emma}, so

$$
c^*(r_t) = \text{Emma}
$$

### Case 8. FT Tie Broken Randomly

**Condition.** Visit counts tie, nearest-gap distances tie, and workloads tie, so the final choice is random.

At month **15**, the examiner is **Tessa**, so the eligible candidates are **Emma, Axie**.

$$
\mathrm{Cand}(r_t) = C \setminus \{e_t\} = \{\text{Emma}, \text{Axie}\}
$$

The examiner-history sets are:

$$
V(\text{Emma}, i_t) = \{9, 27\}\qquad V(\text{Axie}, i_t) = \{3, 21\}
$$

Therefore the contact counts are:

$$
n(\text{Emma}, i_t) = 2\qquad n(\text{Axie}, i_t) = 2
$$

The Never Seen set is:

$$
\mathrm{NS}(r_t) = \varnothing
$$

Because no candidate is fully blind, we move to the Least Visits rule:

$$
n^*(r_t) = \min(2, 2) = 2
$$

$$
\mathrm{LV}(r_t) = \{\text{Emma}, \text{Axie}\}
$$

The Least Visits set is still tied, so we compare nearest-gap distances:

$$
d(\text{Emma} \mid r_t) = \min(\lvert 15-9 \rvert, \lvert 15-27 \rvert) = 6
$$

$$
d(\text{Axie} \mid r_t) = \min(\lvert 15-3 \rvert, \lvert 15-21 \rvert) = 6
$$

The nearest-gap distances are tied, so workload is used next:

$$
W_t(\text{Emma}) = 3\qquad W_t(\text{Axie}) = 3
$$

The workloads are also tied, so this is a perfect final tie. With seed = 42, the reproducible random choice is **Axie**.

$$
c^*(r_t) = \text{Axie}
$$

## Jessie Working example

This section uses **only** the two example children from Jessie’s original message: **1108** and **1120**.

The exact rule scenarios that appear in these rows are:

1. **Unique Never Seen (NS)**
2. **Least Visits (LV)**
3. **Furthest in Time (FT)**
4. **Missing visit (NA)**

The workload tie-break and random tie-break cases do **not** occur in Jessie’s exact data, so they are not forced here.

The key formulas used in this example are:

$$
\mathrm{Cand}(r_t)=C \setminus \{e_t\}
$$

$$
\mathrm{NS}(r_t)=\{c \in \mathrm{Cand}(r_t): n(c,i_t)=0\}
$$

$$
n^*(r_t)=\min_{c \in \mathrm{Cand}(r_t)} n(c,i_t),
\qquad
\mathrm{LV}(r_t)=\{c \in \mathrm{Cand}(r_t): n(c,i_t)=n^*(r_t)\}
$$

$$
d(c \mid r_t)=\min_{v \in V(c,i_t)} \lvert m_t-v \rvert,
\qquad
c^*(r_t)=\operatorname*{argmax}_{c \in \mathrm{LV}(r_t)} d(c \mid r_t)
$$

In [28]:
# Jessie Working example: use only the two original examples exactly as provided.
# Output table: ID, Visit Month, Examiner, Assigned Scoring Clinician.
# Observed scenarios in these rows: NS, LV, FT, and NA.

jessie_example_df = pd.DataFrame(
    [
        {"ID": 1108, "Visit Month": 9, "Examiner": "Tessa"},
        {"ID": 1108, "Visit Month": 12, "Examiner": "Tessa"},
        {"ID": 1108, "Visit Month": 15, "Examiner": "Emma"},
        {"ID": 1108, "Visit Month": 18, "Examiner": "Tessa"},
        {"ID": 1108, "Visit Month": 24, "Examiner": "Emma"},
        {"ID": 1120, "Visit Month": 9, "Examiner": "Axie"},
        {"ID": 1120, "Visit Month": 12, "Examiner": "Tessa"},
        {"ID": 1120, "Visit Month": 15, "Examiner": "NA"},
        {"ID": 1120, "Visit Month": 18, "Examiner": "Tessa"},
        {"ID": 1120, "Visit Month": 24, "Examiner": "Emma"},
    ]
)

jessie_assigned = assign_scoring_clinicians(jessie_example_df, seed=42)

jessie_output = jessie_assigned[
    ["ID", "Visit Month", "Examiner", "Assigned Scoring Clinician"]
]

expected_jessie = pd.DataFrame(
    [
        {"ID": 1108, "Visit Month": 9, "Examiner": "Tessa", "Assigned Scoring Clinician": "Axie"},
        {"ID": 1108, "Visit Month": 12, "Examiner": "Tessa", "Assigned Scoring Clinician": "Axie"},
        {"ID": 1108, "Visit Month": 15, "Examiner": "Emma", "Assigned Scoring Clinician": "Axie"},
        {"ID": 1108, "Visit Month": 18, "Examiner": "Tessa", "Assigned Scoring Clinician": "Axie"},
        {"ID": 1108, "Visit Month": 24, "Examiner": "Emma", "Assigned Scoring Clinician": "Axie"},
        {"ID": 1120, "Visit Month": 9, "Examiner": "Axie", "Assigned Scoring Clinician": "Emma"},
        {"ID": 1120, "Visit Month": 12, "Examiner": "Tessa", "Assigned Scoring Clinician": "Emma"},
        {"ID": 1120, "Visit Month": 15, "Examiner": "NA", "Assigned Scoring Clinician": "NA"},
        {"ID": 1120, "Visit Month": 18, "Examiner": "Tessa", "Assigned Scoring Clinician": "Axie"},
        {"ID": 1120, "Visit Month": 24, "Examiner": "Emma", "Assigned Scoring Clinician": "Axie"},
    ]
)

assert jessie_output.reset_index(drop=True).equals(expected_jessie)

display(jessie_output)

jessie_decisions_markdown = "\n".join([
    "### Decision scenarios observed in Jessie's exact data",
    "",
    "| Observed scenario | Where it happens | Condition | Decision |",
    "| --- | --- | --- | --- |",
    r"| Unique NS | Child 1108 at 9m, 12m, 15m, 18m, 24m | $\mathrm{NS}(r_t)=\{\text{Axie}\}$ | Axie is assigned because Axie never examined child 1108. |",
    r"| Unique LV | Child 1120 at 9m and 24m | $\mathrm{NS}(r_t)=\varnothing$ and $\mathrm{LV}(r_t)$ has one winner | Emma wins at 9m and Axie wins at 24m because each has fewer prior contacts than the other candidate. |",
    r"| Unique FT | Child 1120 at 12m and 18m | $\mathrm{LV}(r_t)=\{\text{Emma},\text{Axie}\}$ and compare $d(c \mid r_t)$ | The candidate whose nearest own visit is further away in time is assigned. |",
    r"| NA visit | Child 1120 at 15m | $e_t=\mathrm{NA}$ | No scorer is assigned. |",
])

display(Markdown(jessie_decisions_markdown))

jessie_summary_markdown = "\n".join([
    "### Simple interpretation of Jessie’s two examples",
    "",
    "- **Child 1108** shows the pure **Never Seen (NS)** rule: Axie never examined this child, so Axie scores every completed visit.",
    "- **Child 1120 at 9m and 24m** shows the **Least Visits (LV)** rule: the winning scorer has fewer prior contacts with the child than the other candidate.",
    "- **Child 1120 at 12m and 18m** shows the **Furthest in Time (FT)** rule: Emma and Axie are tied on visit count, so the scorer is whichever one saw the child furthest away in age from the current visit.",
    "- **Child 1120 at 15m** shows the **NA** rule: no visit occurred, so no scorer is assigned.",
    "",
    "These two examples reproduce Jessie’s expected final 4-column output exactly.",
])

display(Markdown(jessie_summary_markdown))

,ID,Visit Month,Examiner,Assigned Scoring Clinician
0,1108,9,Tessa,Axie
1,1108,12,Tessa,Axie
2,1108,15,Emma,Axie
3,1108,18,Tessa,Axie
4,1108,24,Emma,Axie
5,1120,9,Axie,Emma
6,1120,12,Tessa,Emma
7,1120,15,NA,NA
8,1120,18,Tessa,Axie
9,1120,24,Emma,Axie


### Decision scenarios observed in Jessie's exact data

| Observed scenario | Where it happens | Condition | Decision |
| --- | --- | --- | --- |
| Unique NS | Child 1108 at 9m, 12m, 15m, 18m, 24m | $\mathrm{NS}(r_t)=\{\text{Axie}\}$ | Axie is assigned because Axie never examined child 1108. |
| Unique LV | Child 1120 at 9m and 24m | $\mathrm{NS}(r_t)=\varnothing$ and $\mathrm{LV}(r_t)$ has one winner | Emma wins at 9m and Axie wins at 24m because each has fewer prior contacts than the other candidate. |
| Unique FT | Child 1120 at 12m and 18m | $\mathrm{LV}(r_t)=\{\text{Emma},\text{Axie}\}$ and compare $d(c \mid r_t)$ | The candidate whose nearest own visit is further away in time is assigned. |
| NA visit | Child 1120 at 15m | $e_t=\mathrm{NA}$ | No scorer is assigned. |

### Simple interpretation of Jessie’s two examples

- **Child 1108** shows the pure **Never Seen (NS)** rule: Axie never examined this child, so Axie scores every completed visit.
- **Child 1120 at 9m and 24m** shows the **Least Visits (LV)** rule: the winning scorer has fewer prior contacts with the child than the other candidate.
- **Child 1120 at 12m and 18m** shows the **Furthest in Time (FT)** rule: Emma and Axie are tied on visit count, so the scorer is whichever one saw the child furthest away in age from the current visit.
- **Child 1120 at 15m** shows the **NA** rule: no visit occurred, so no scorer is assigned.

These two examples reproduce Jessie’s expected final 4-column output exactly.

## 10. IPSA REDCap Export Pipeline

This section adds a safe, reproducible, read-only pipeline for exporting IPSA CSBS data from REDCap and applying the blind-scoring clinician assignment algorithm.

### Security rules enforced here

- The REDCap API token is read **only** from the `REDCAP_API_TOKEN` environment variable.
- The token is never hard-coded into this notebook, never printed, never written to disk, and never stored in the run manifest.
- Only read-only REDCap API operations are used: an optional connectivity check, metadata export, instrument export, form-event mapping export, repeating metadata export, and record export.
- Output files are written locally only; nothing is written back to REDCap.

### Suggested run order

1. Review and fill the editable `CONFIG` block.
2. Run the setup cell that defines the REDCap pipeline helpers.
3. Run the synthetic test cell to confirm the assignment logic still behaves as expected.
4. Set `REDCAP_API_TOKEN` in your shell or notebook kernel.
5. Run the final live pipeline cell to export IPSA, build audits, write local files, and preview the first 20 assigned rows.

In [29]:
import json
import logging
import os
import re
import sys
from datetime import datetime, timezone
from pathlib import Path
from urllib.parse import urlparse

import requests
from IPython.display import Markdown, display

# -----------------------------
# CONFIG: edit these values after running metadata discovery.
# -----------------------------
API_URL = "https://redcap.research.sc.edu/api/"
TOKEN_ENV_VAR = "REDCAP_API_TOKEN"
OUTPUT_DIR = Path("csbs_redcap_outputs")
REQUEST_TIMEOUT = (10, 60)
PIPELINE_RANDOM_SEED = 42
PREVIEW_ROWS = 20
RUN_CONNECTIVITY_CHECK = True

RECORD_ID_FIELD = None
CSBS_FORM_NAMES = []
MONTH_FIELD = None
EXAMINER_FIELD = None
COMPLETION_FIELD = None
COMPLETE_STATUS_VALUE = "2"
CSBS_FORM_INDICATOR_FIELDS = []
EVENT_FIELD_HANDLING = "retain redcap_event_name if present"
REPEATING_INSTANCE_HANDLING = "retain redcap_repeat_instrument and redcap_repeat_instance if present"
JESSI_RECORD_ID = "REPLACE_WITH_JESSI_RECORD_ID"
ALLOW_BLANK_EXAMINER_RANDOM_ASSIGNMENT = False

FIELD_DISCOVERY_TERMS = ("csbs", "csbs-bs", "examiner", "month", "age", "completion")
RULE_LABEL_MAP = {
    "NS": "NS",
    "NS + lower workload": "NS lower workload",
    "NS + random after workload tie": "NS random after workload tie",
    "LV": "LV",
    "FT": "FT",
    "FT + lower workload": "FT lower workload",
    "FT + random after workload tie": "FT random after workload tie",
    "NA visit": "NA visit",
}

LOGGER = logging.getLogger("csbs_redcap_pipeline")
if not LOGGER.handlers:
    logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")

In [30]:
def _safe_text(value):
    if value is None or pd.isna(value):
        return ""
    return str(value).strip()

def _normalize_text(value):
    return _safe_text(value).casefold()

def _contains_any(value, terms):
    text = _normalize_text(value)
    return any(term in text for term in terms)

def _sanitize_redcap_text(value, max_length=400):
    text = " ".join(str(value).replace("\n", " ").split())
    text = re.sub(r"\b[a-fA-F0-9]{24,128}\b", "[redacted]", text)
    text = re.sub(
        r"(?i)(token|api[_ -]?key|authorization)(\s*[:=]\s*)([^\s,;]+)",
        r"\1\2[redacted]",
        text,
    )
    return text[:max_length] + ("…" if len(text) > max_length else "")

def _build_payload(token, content, scalar_params=None, list_params=None):
    payload = [
        ("token", token),
        ("content", content),
        ("format", "json"),
        ("returnFormat", "json"),
    ]
    for key, value in (scalar_params or {}).items():
        if value is not None:
            payload.append((key, str(value)))
    for key, values in (list_params or {}).items():
        for index, value in enumerate(values):
            payload.append((f"{key}[{index}]", str(value)))
    return payload

def redcap_post(token, content, scalar_params=None, list_params=None):
    response = requests.post(
        API_URL,
        data=_build_payload(token, content, scalar_params, list_params),
        timeout=REQUEST_TIMEOUT,
    )
    try:
        response.raise_for_status()
    except requests.HTTPError as exc:
        body = _sanitize_redcap_text(response.text)
        raise RuntimeError(
            f"HTTP error for content={content!r}: {response.status_code}. {body}"
        ) from exc

    try:
        data = response.json()
    except ValueError as exc:
        body = _sanitize_redcap_text(response.text)
        raise RuntimeError(
            f"Non-JSON REDCap response for content={content!r}: {body}"
        ) from exc

    if isinstance(data, dict) and data.get("error"):
        raise RuntimeError(
            f"REDCAP API error for content={content!r}: {_sanitize_redcap_text(data['error'])}"
        )

    return data

def export_metadata_df(token):
    return pd.DataFrame(redcap_post(token, "metadata"))

def export_instruments_df(token):
    return pd.DataFrame(redcap_post(token, "instrument"))

def export_form_event_mapping_df(token):
    try:
        return pd.DataFrame(redcap_post(token, "formEventMapping"))
    except Exception as exc:
        LOGGER.warning("formEventMapping export unavailable: %s", _sanitize_redcap_text(exc))
        return pd.DataFrame()

def export_repeating_forms_events_df(token):
    try:
        return pd.DataFrame(redcap_post(token, "repeatingFormsEvents"))
    except Exception as exc:
        LOGGER.warning("repeatingFormsEvents export unavailable: %s", _sanitize_redcap_text(exc))
        return pd.DataFrame()

def connectivity_check(token):
    if not RUN_CONNECTIVITY_CHECK:
        return {}
    try:
        return redcap_post(token, "project")
    except Exception as exc:
        raise RuntimeError(
            "Connectivity check failed. Confirm REDCAP_API_TOKEN is set correctly and has read-only access. "
            + _sanitize_redcap_text(exc)
        ) from exc

def _candidate_roles_from_row(row):
    field_name = _normalize_text(row.get("field_name", ""))
    field_label = _normalize_text(row.get("field_label", ""))
    form_name = _normalize_text(row.get("form_name", ""))
    field_type = _normalize_text(row.get("field_type", ""))
    combined = " ".join([field_name, field_label, form_name, field_type])
    roles = []

    if field_name in {"record_id", "study_id", "participant_id"} or "record id" in field_label:
        roles.append("ID")
    if any(term in combined for term in ("csbs", "csbs-bs")):
        roles.append("CSBS indicator")
    if any(term in combined for term in ("examiner", "clinician", "assessor")):
        roles.append("examiner")
    if any(term in combined for term in ("visit month", "month", "age at visit", "visit age")):
        roles.append("visit month")
    if field_name.endswith("_complete") or "complete" in field_label or "completion" in combined:
        roles.append("completion")

    return "; ".join(dict.fromkeys(roles))

def build_field_mapping_audit(metadata_df):
    if metadata_df.empty:
        return pd.DataFrame(
            columns=[
                "field_name",
                "field_label",
                "form_name",
                "field_type",
                "candidate_semantic_role",
            ]
        )

    audit_df = metadata_df.copy()
    for column in ("field_name", "field_label", "form_name", "field_type"):
        if column not in audit_df.columns:
            audit_df[column] = ""
        audit_df[column] = audit_df[column].map(_safe_text)

    audit_df["candidate_semantic_role"] = audit_df.apply(_candidate_roles_from_row, axis=1)
    relevant_mask = audit_df.apply(
        lambda row: bool(row["candidate_semantic_role"])
        or _contains_any(row["field_name"], FIELD_DISCOVERY_TERMS)
        or _contains_any(row["field_label"], FIELD_DISCOVERY_TERMS)
        or _contains_any(row["form_name"], FIELD_DISCOVERY_TERMS),
        axis=1,
    )

    audit_df = audit_df.loc[relevant_mask, [
        "field_name",
        "field_label",
        "form_name",
        "field_type",
        "candidate_semantic_role",
    ]].sort_values(["form_name", "field_name"], kind="stable").reset_index(drop=True)

    return audit_df

def validate_pipeline_config(metadata_df):
    missing = []
    required_scalars = {
        "RECORD_ID_FIELD": RECORD_ID_FIELD,
        "MONTH_FIELD": MONTH_FIELD,
        "EXAMINER_FIELD": EXAMINER_FIELD,
        "COMPLETION_FIELD": COMPLETION_FIELD,
        "COMPLETE_STATUS_VALUE": COMPLETE_STATUS_VALUE,
    }
    for key, value in required_scalars.items():
        if value in (None, "", []):
            missing.append(key)

    if not CSBS_FORM_NAMES and not CSBS_FORM_INDICATOR_FIELDS:
        missing.append("CSBS_FORM_NAMES or CSBS_FORM_INDICATOR_FIELDS")

    if missing:
        raise ValueError(
            "Fill the CONFIG block before running the live export. Missing: " + ", ".join(missing)
        )

    available_fields = set(metadata_df.get("field_name", pd.Series(dtype=str)).astype(str))
    for field_name in [RECORD_ID_FIELD, MONTH_FIELD, EXAMINER_FIELD, COMPLETION_FIELD, *CSBS_FORM_INDICATOR_FIELDS]:
        if field_name and field_name not in available_fields:
            raise ValueError(f"Configured field not found in metadata: {field_name}")

    available_forms = set(metadata_df.get("form_name", pd.Series(dtype=str)).astype(str))
    for form_name in CSBS_FORM_NAMES:
        if form_name not in available_forms:
            raise ValueError(f"Configured form not found in metadata: {form_name}")

def build_record_export_field_list():
    fields = [
        RECORD_ID_FIELD,
        MONTH_FIELD,
        EXAMINER_FIELD,
        COMPLETION_FIELD,
        *CSBS_FORM_INDICATOR_FIELDS,
        "redcap_event_name",
        "redcap_repeat_instrument",
        "redcap_repeat_instance",
    ]
    return [field for field in dict.fromkeys(fields) if field]

def export_csbs_candidate_records_df(token):
    scalar_params = {
        "type": "flat",
        "rawOrLabel": "raw",
        "rawOrLabelHeaders": "raw",
        "exportCheckboxLabel": "false",
        "exportDataAccessGroups": "false",
    }
    list_params = {"fields": build_record_export_field_list()}
    if CSBS_FORM_NAMES:
        list_params["forms"] = CSBS_FORM_NAMES
    return pd.DataFrame(redcap_post(token, "record", scalar_params, list_params))

def _row_is_csbs_relevant(row):
    if CSBS_FORM_NAMES:
        return True
    return any(_safe_text(row.get(field_name, "")) for field_name in CSBS_FORM_INDICATOR_FIELDS)

def _is_complete_status(value):
    if pd.isna(value):
        return False
    return str(value).strip() == str(COMPLETE_STATUS_VALUE)

def _canonicalize_examiner_for_pipeline(value):
    if pd.isna(value):
        return pd.NA, None
    text = str(value).strip()
    if not text or text.casefold() in {"na", "n/a"}:
        return pd.NA, None
    allowed = {clinician.casefold(): clinician for clinician in CLINICIANS}
    canonical = allowed.get(text.casefold())
    if canonical is None:
        return pd.NA, f"unexpected examiner value: {text}"
    return canonical, None

def _coerce_visit_month(value):
    numeric = pd.to_numeric([value], errors="coerce")[0]
    if pd.isna(numeric):
        return pd.NA
    if float(numeric).is_integer():
        return int(numeric)
    return float(numeric)

def normalize_csbs_records(raw_records_df):
    included_rows = []
    exclusions = []
    csbs_filtered_rows = 0
    completed_rows = 0

    for _, row in raw_records_df.iterrows():
        record_id = row.get(RECORD_ID_FIELD, pd.NA)
        event_name = row.get("redcap_event_name", pd.NA)
        repeat_instrument = row.get("redcap_repeat_instrument", pd.NA)
        repeat_instance = row.get("redcap_repeat_instance", pd.NA)

        base_context = {
            "ID": _safe_text(record_id),
            "Visit Month": row.get(MONTH_FIELD, pd.NA),
            "Examiner": row.get(EXAMINER_FIELD, pd.NA),
            "redcap_event_name": event_name,
            "redcap_repeat_instrument": repeat_instrument,
            "redcap_repeat_instance": repeat_instance,
        }

        if not _row_is_csbs_relevant(row):
            exclusions.append({**base_context, "Exclusion Reason": "not a CSBS form"})
            continue
        csbs_filtered_rows += 1

        if not _is_complete_status(row.get(COMPLETION_FIELD, pd.NA)):
            exclusions.append({**base_context, "Exclusion Reason": "incomplete CSBS form"})
            continue
        completed_rows += 1

        visit_month = _coerce_visit_month(row.get(MONTH_FIELD, pd.NA))
        if pd.isna(visit_month):
            exclusions.append({**base_context, "Exclusion Reason": "missing visit month"})
            continue

        examiner, examiner_error = _canonicalize_examiner_for_pipeline(row.get(EXAMINER_FIELD, pd.NA))
        if examiner_error:
            exclusions.append({**base_context, "Exclusion Reason": "unexpected examiner value", "Exclusion Detail": examiner_error})
            continue

        if pd.isna(examiner):
            if ALLOW_BLANK_EXAMINER_RANDOM_ASSIGNMENT:
                raise NotImplementedError(
                    "Blank-examiner random assignment is intentionally disabled in this notebook section. "
                    "Keep ALLOW_BLANK_EXAMINER_RANDOM_ASSIGNMENT=False unless you extend the assignment function with a governed policy."
                )
            exclusions.append({**base_context, "Exclusion Reason": "missing examiner"})
            continue

        included_rows.append(
            {
                "ID": _safe_text(record_id),
                "Visit Month": visit_month,
                "Examiner": examiner,
                "redcap_event_name": event_name,
                "redcap_repeat_instrument": repeat_instrument,
                "redcap_repeat_instance": repeat_instance,
            }
        )

    eligible_df = pd.DataFrame(included_rows)
    exclusions_df = pd.DataFrame(exclusions)

    duplicate_count = 0
    if not eligible_df.empty:
        duplicate_mask = eligible_df.duplicated(["ID", "Visit Month"], keep=False)
        duplicate_count = int(duplicate_mask.sum())
        if duplicate_mask.any():
            duplicate_rows = eligible_df.loc[duplicate_mask].copy()
            duplicate_rows["Exclusion Reason"] = "duplicate visit requiring review"
            exclusions_df = pd.concat([exclusions_df, duplicate_rows], ignore_index=True, sort=False)
            eligible_df = eligible_df.loc[~duplicate_mask].copy()

    counts = {
        "exported_rows": int(len(raw_records_df)),
        "csbs_filtered_rows": int(csbs_filtered_rows),
        "completed_rows": int(completed_rows),
        "eligible_rows": int(len(eligible_df)),
        "excluded_rows": int(len(exclusions_df)),
        "duplicate_rows": int(duplicate_count),
    }

    return {
        "eligible_df": eligible_df.reset_index(drop=True),
        "exclusions_df": exclusions_df.reset_index(drop=True),
        "counts": counts,
    }

def _apply_rule_label_map(frame, column_name):
    result = frame.copy()
    result[column_name] = result[column_name].map(lambda value: RULE_LABEL_MAP.get(value, value))
    return result

def build_assignment_outputs(eligible_df):
    if eligible_df.empty:
        assignments_df = pd.DataFrame(
            columns=[
                "ID",
                "Visit Month",
                "Examiner",
                "Assigned Scoring Clinician",
                "Assignment Rule",
                "Assigned Clinician Visit Count",
                "Assigned Clinician Nearest Gap",
                "Workload Before Assignment",
                "redcap_event_name",
                "redcap_repeat_instrument",
                "redcap_repeat_instance",
            ]
        )
        candidate_trace_df = pd.DataFrame()
        workload_df = pd.DataFrame(columns=["Clinician", "Assignments"])
        return assignments_df, candidate_trace_df, workload_df

    sort_columns = [
        "ID",
        "Visit Month",
        "redcap_event_name",
        "redcap_repeat_instrument",
        "redcap_repeat_instance",
    ]
    eligible_sorted = eligible_df.copy()
    for column in sort_columns[2:]:
        if column not in eligible_sorted.columns:
            eligible_sorted[column] = ""
    eligible_sorted = eligible_sorted.sort_values(sort_columns, kind="stable").reset_index(drop=True)

    assigned_df, candidate_trace_df = assign_scoring_clinicians(
        eligible_sorted,
        clinicians=CLINICIANS,
        seed=PIPELINE_RANDOM_SEED,
        return_trace=True,
    )

    assigned_df = _apply_rule_label_map(assigned_df, "Assignment Rule")
    candidate_trace_df = _apply_rule_label_map(candidate_trace_df, "Winning Rule")

    context_df = eligible_sorted[[
        "ID",
        "Visit Month",
        "Examiner",
        "redcap_event_name",
        "redcap_repeat_instrument",
        "redcap_repeat_instance",
    ]].drop_duplicates()

    candidate_trace_df = candidate_trace_df.merge(
        context_df,
        on=["ID", "Visit Month", "Examiner"],
        how="left",
        validate="many_to_one",
    )

    assignments_df = assigned_df[[
        "ID",
        "Visit Month",
        "Examiner",
        "Assigned Scoring Clinician",
        "Assignment Rule",
        "Assigned Clinician Visit Count",
        "Assigned Clinician Nearest Gap",
        "Workload Before Assignment",
        "redcap_event_name",
        "redcap_repeat_instrument",
        "redcap_repeat_instance",
    ]].copy()

    workload_series = workload_summary(assignments_df)
    if isinstance(workload_series, pd.DataFrame):
        workload_df = workload_series.reset_index()
    else:
        workload_df = pd.DataFrame(columns=["Clinician", "Assignments"])

    return assignments_df, candidate_trace_df, workload_df

def run_quality_checks(assignments_df, candidate_trace_df, workload_df, counts):
    checks = {
        "no_assigned_scorer_equals_examiner": bool(
            assignments_df.empty or (assignments_df["Assigned Scoring Clinician"] != assignments_df["Examiner"]).all()
        ),
        "every_included_row_has_valid_numeric_visit_month": bool(
            assignments_df.empty or pd.to_numeric(assignments_df["Visit Month"], errors="coerce").notna().all()
        ),
        "every_nonmissing_examiner_is_recognized": bool(
            assignments_df.empty or assignments_df["Examiner"].isin(CLINICIANS).all()
        ),
        "every_included_row_receives_exactly_one_scorer": bool(
            assignments_df.empty or assignments_df["Assigned Scoring Clinician"].isin(CLINICIANS).all()
        ),
        "assignment_count_equals_number_of_included_visits": int(len(assignments_df)) == int(counts["eligible_rows"]),
        "workload_total_equals_number_of_included_visits": int(workload_df.get("Assignments", pd.Series(dtype=int)).sum()) == int(len(assignments_df)),
        "candidate_trace_has_two_rows_per_assigned_visit": int(len(candidate_trace_df)) == int(2 * len(assignments_df)),
        "duplicate_rows_flagged_for_review": int(counts["duplicate_rows"]) >= 0,
    }

    failed = [name for name, passed in checks.items() if not passed]
    if failed:
        raise AssertionError("Quality-control checks failed: " + ", ".join(failed))

    return checks

def build_run_manifest(mapping_audit_df, counts, checks):
    return {
        "generated_at_utc": datetime.now(timezone.utc).isoformat(),
        "endpoint_hostname": urlparse(API_URL).hostname,
        "selected_forms": CSBS_FORM_NAMES,
        "selected_fields": {
            "record_id": RECORD_ID_FIELD,
            "visit_month": MONTH_FIELD,
            "examiner": EXAMINER_FIELD,
            "completion": COMPLETION_FIELD,
            "indicator_fields": CSBS_FORM_INDICATOR_FIELDS,
        },
        "event_field_handling": EVENT_FIELD_HANDLING,
        "repeating_instance_handling": REPEATING_INSTANCE_HANDLING,
        "random_seed": PIPELINE_RANDOM_SEED,
        "row_counts": counts,
        "quality_checks": checks,
        "mapping_audit_rows": int(len(mapping_audit_df)),
        "version_metadata": {
            "python": sys.version.split()[0],
            "pandas": pd.__version__,
            "requests": requests.__version__,
        },
    }

def write_pipeline_outputs(
    assignments_df,
    candidate_trace_df,
    exclusions_df,
    workload_df,
    mapping_audit_df,
    manifest,
    output_dir=OUTPUT_DIR,
):
    output_dir.mkdir(parents=True, exist_ok=True)
    assignments_df.to_csv(output_dir / "csbs_scoring_assignments.csv", index=False)
    candidate_trace_df.to_csv(output_dir / "csbs_scoring_candidate_trace.csv", index=False)
    exclusions_df.to_csv(output_dir / "csbs_exclusions_audit.csv", index=False)
    workload_df.to_csv(output_dir / "csbs_workload_summary.csv", index=False)
    mapping_audit_df.to_csv(output_dir / "csbs_field_mapping_audit.csv", index=False)
    with (output_dir / "csbs_run_manifest.json").open("w", encoding="utf-8") as handle:
        json.dump(manifest, handle, indent=2)

def run_jessi_validation(assignments_df, candidate_trace_df, jessi_record_id=JESSI_RECORD_ID, expected_assignments_df=None):
    if not jessi_record_id or "REPLACE_WITH_JESSI_RECORD_ID" in str(jessi_record_id):
        print("Jessi validation skipped: set JESSI_RECORD_ID in the CONFIG block to run dataset-level validation.")
        return None

    jessi_id = str(jessi_record_id)
    jessi_rows = assignments_df.loc[assignments_df["ID"].astype(str) == jessi_id, [
        "ID",
        "Visit Month",
        "Examiner",
        "Assigned Scoring Clinician",
        "Assignment Rule",
        "Assigned Clinician Visit Count",
        "Assigned Clinician Nearest Gap",
        "Workload Before Assignment",
    ]].reset_index(drop=True)

    jessi_trace = candidate_trace_df.loc[candidate_trace_df["ID"].astype(str) == jessi_id, [
        "ID",
        "Visit Month",
        "Examiner",
        "Candidate",
        "Candidate Visit Count n(c,i)",
        "Never Seen NS",
        "Least Visits LV",
        "Furthest in Time FT",
        "Nearest Gap d(c|r_t)",
        "Workload Before W_t(c)",
        "Selected",
        "Winning Rule",
    ]].reset_index(drop=True)

    display(jessi_rows)
    display(jessi_trace)

    if expected_assignments_df is None:
        print("Jessi validation trace displayed. No pass/fail assertion was made because no expected assignment table was supplied.")
        return {"jessi_rows": jessi_rows, "jessi_trace": jessi_trace, "validated": False}

    actual = jessi_rows[["ID", "Visit Month", "Examiner", "Assigned Scoring Clinician"]].reset_index(drop=True)
    expected = expected_assignments_df[["ID", "Visit Month", "Examiner", "Assigned Scoring Clinician"]].reset_index(drop=True)
    validated = actual.equals(expected)
    print(f"Jessi validation {'passed' if validated else 'did not pass'}.")
    return {"jessi_rows": jessi_rows, "jessi_trace": jessi_trace, "validated": validated}

def print_non_phi_run_report(counts, checks, output_dir=OUTPUT_DIR):
    lines = [
        "CSBS IPSA pipeline run report",
        f"- Exported rows: {counts['exported_rows']}",
        f"- CSBS-filtered rows: {counts['csbs_filtered_rows']}",
        f"- Completed rows: {counts['completed_rows']}",
        f"- Eligible rows: {counts['eligible_rows']}",
        f"- Excluded rows: {counts['excluded_rows']}",
        f"- Duplicate rows flagged: {counts['duplicate_rows']}",
        f"- Assigned rows: {counts['assigned_rows']}",
        f"- Output directory: {output_dir}",
        f"- All QC checks passed: {all(checks.values())}",
    ]
    print("\n".join(lines))

def run_pipeline_synthetic_tests():
    synthetic_df = pd.DataFrame(
        [
            {"ID": 1108, "Visit Month": 9, "Examiner": "Tessa"},
            {"ID": 1108, "Visit Month": 12, "Examiner": "Tessa"},
            {"ID": 1108, "Visit Month": 15, "Examiner": "Emma"},
            {"ID": 1108, "Visit Month": 18, "Examiner": "Tessa"},
            {"ID": 1108, "Visit Month": 24, "Examiner": "Emma"},
            {"ID": 1120, "Visit Month": 9, "Examiner": "Axie"},
            {"ID": 1120, "Visit Month": 12, "Examiner": "Tessa"},
            {"ID": 1120, "Visit Month": 15, "Examiner": "NA"},
            {"ID": 1120, "Visit Month": 18, "Examiner": "Tessa"},
            {"ID": 1120, "Visit Month": 24, "Examiner": "Emma"},
        ]
    )
    actual_df = assign_scoring_clinicians(synthetic_df, clinicians=CLINICIANS, seed=PIPELINE_RANDOM_SEED)
    actual_df = _apply_rule_label_map(actual_df, "Assignment Rule")

    expected_rows = [
        (1108, 9, "Tessa", "Axie", "NS"),
        (1108, 12, "Tessa", "Axie", "NS"),
        (1108, 15, "Emma", "Axie", "NS"),
        (1108, 18, "Tessa", "Axie", "NS"),
        (1108, 24, "Emma", "Axie", "NS"),
        (1120, 9, "Axie", "Emma", "LV"),
        (1120, 12, "Tessa", "Emma", "FT"),
        (1120, 15, "NA", "NA", "NA visit"),
        (1120, 18, "Tessa", "Axie", "FT"),
        (1120, 24, "Emma", "Axie", "LV"),
    ]
    expected_df = pd.DataFrame(
        expected_rows,
        columns=["ID", "Visit Month", "Examiner", "Assigned Scoring Clinician", "Assignment Rule"],
    )

    actual_comp = actual_df[[
        "ID",
        "Visit Month",
        "Examiner",
        "Assigned Scoring Clinician",
        "Assignment Rule",
    ]].reset_index(drop=True)

    if not actual_comp.equals(expected_df):
        raise AssertionError("Synthetic CSBS assignment tests failed. Review the current assignment function before exporting live REDCap data.")

    print("Synthetic CSBS assignment tests passed.")
    display(actual_comp)
    return actual_comp

def run_ipsa_csbs_pipeline(write_outputs=True):
    token = os.getenv(TOKEN_ENV_VAR)
    if not token:
        raise EnvironmentError(
            f"{TOKEN_ENV_VAR} is not set. Export the token in your shell or notebook kernel before running the live IPSA pipeline."
        )

    connectivity = connectivity_check(token)
    metadata_df = export_metadata_df(token)
    instruments_df = export_instruments_df(token)
    form_event_mapping_df = export_form_event_mapping_df(token)
    repeating_df = export_repeating_forms_events_df(token)
    mapping_audit_df = build_field_mapping_audit(metadata_df)

    display(mapping_audit_df)
    validate_pipeline_config(metadata_df)

    raw_records_df = export_csbs_candidate_records_df(token)
    normalized = normalize_csbs_records(raw_records_df)
    assignments_df, candidate_trace_df, workload_df = build_assignment_outputs(normalized["eligible_df"])

    counts = dict(normalized["counts"])
    counts["assigned_rows"] = int(len(assignments_df))

    checks = run_quality_checks(assignments_df, candidate_trace_df, workload_df, counts)
    manifest = build_run_manifest(mapping_audit_df, counts, checks)

    if write_outputs:
        write_pipeline_outputs(
            assignments_df,
            candidate_trace_df,
            normalized["exclusions_df"],
            workload_df,
            mapping_audit_df,
            manifest,
        )

    print_non_phi_run_report(counts, checks)
    display(assignments_df.head(PREVIEW_ROWS))

    return {
        "connectivity": connectivity,
        "metadata": metadata_df,
        "instruments": instruments_df,
        "form_event_mapping": form_event_mapping_df,
        "repeating_forms_events": repeating_df,
        "field_mapping_audit": mapping_audit_df,
        "raw_records": raw_records_df,
        "eligible_assignments": assignments_df,
        "candidate_trace": candidate_trace_df,
        "exclusions": normalized["exclusions_df"],
        "workload_summary": workload_df,
        "manifest": manifest,
        "quality_checks": checks,
    }

In [31]:
pipeline_synthetic_test_output = run_pipeline_synthetic_tests()
pipeline_synthetic_test_output.head(20)

Synthetic CSBS assignment tests passed.


,ID,Visit Month,Examiner,Assigned Scoring Clinician,Assignment Rule
0,1108,9,Tessa,Axie,NS
1,1108,12,Tessa,Axie,NS
2,1108,15,Emma,Axie,NS
3,1108,18,Tessa,Axie,NS
4,1108,24,Emma,Axie,NS
5,1120,9,Axie,Emma,LV
6,1120,12,Tessa,Emma,FT
7,1120,15,NA,NA,NA visit
8,1120,18,Tessa,Axie,FT
9,1120,24,Emma,Axie,LV


,ID,Visit Month,Examiner,Assigned Scoring Clinician,Assignment Rule
0,1108,9,Tessa,Axie,NS
1,1108,12,Tessa,Axie,NS
2,1108,15,Emma,Axie,NS
3,1108,18,Tessa,Axie,NS
4,1108,24,Emma,Axie,NS
5,1120,9,Axie,Emma,LV
6,1120,12,Tessa,Emma,FT
7,1120,15,NA,NA,NA visit
8,1120,18,Tessa,Axie,FT
9,1120,24,Emma,Axie,LV


In [32]:
# Live IPSA run: execute this cell only after reviewing CONFIG and setting REDCAP_API_TOKEN.
if os.getenv(TOKEN_ENV_VAR):
    ipsa_pipeline_results = run_ipsa_csbs_pipeline(write_outputs=True)
else:
    print(
        f"{TOKEN_ENV_VAR} is not set in the current environment. "
        "Review the CONFIG block, export the token locally, and rerun this cell when ready."
    )

REDCAP_API_TOKEN is not set in the current environment. Review the CONFIG block, export the token locally, and rerun this cell when ready.
